# molhiv-gnn — does a GNN beat counting substructures?

Three ways to predict HIV inhibition from a molecule's 2D structure, on the
**same official scaffold split**, scored by the **same OGB evaluator**:

| Arm | Method | Learning |
|---|---|---|
| 1 | Morgan fingerprints + Random Forest | none (chemistry prior) |
| 2 | WEGL: diffusion + optimal-transport embedding + boosted trees | none (structure only) |
| 3 | PNA graph neural network | trained end to end |

**Before running:** Runtime → Change runtime type → **A100** (or T4; the
reference timings below are for a T4, an A100 is faster).

Run the cells in order. Arms 1 and 2 are CPU and finish in minutes; arm 3 needs
the GPU and is the long one (~25 min per seed on a T4). Arm 3 writes its
predictions and checkpoint after **each** seed, so a disconnect part-way still
leaves usable results and `src.report` will table whatever finished.

Everything below is capped at roughly **half the machine's cores** (section 3),
so the run leaves headroom instead of saturating the box.

## 1. Check the GPU

In [ ]:
# Arm 3 is hopeless without a GPU, so fail loudly here rather than 3 hours in.
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')

## 2. Install dependencies

In [ ]:
# torch is already on Colab; these are the extras. The compiled torch-scatter /
# torch-sparse add-ons are NOT needed - PyG >= 2.3 uses native scatter ops.
!pip -q install torch_geometric ogb rdkit POT

## 3. Cap resource usage to ~50%

Sets the thread limits every later cell inherits, and computes the `n_jobs` the
tree arms will use. The thread caps have to be set as environment variables
*before* the worker processes start, which is why they live in their own cell:
OpenMP reads them at library init, so setting them later has no effect.

A note on what can actually be capped. The CPU side is real and worth limiting —
the Random Forest and the boosted trees will otherwise take every core. The GPU
side is not a meaningful knob here: PNA on molhiv at batch 128 is a small model
and will use only a couple of GB of an A100's 40 GB no matter what, so there is
nothing to cap. I have not added a GPU memory fraction limit, because it would
suggest a constraint that does not bind.

In [ ]:
import os
TOTAL = os.cpu_count() or 2
NJOBS = max(1, TOTAL // 2)
print('logical cores:', TOTAL, '-> using', NJOBS)

for var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
            'NUMEXPR_NUM_THREADS'):
    os.environ[var] = str(NJOBS)
    print(var, '=', os.environ[var])

## 4. Upload and unzip the project

Upload `molhiv-gnn.zip` with the file browser on the left (folder icon), then run
this cell. The zip already contains a top-level `molhiv-gnn/` folder.

In [ ]:
import os
assert os.path.exists('molhiv-gnn.zip'), \
    'Upload molhiv-gnn.zip first (folder icon in the left sidebar).'
!unzip -qo molhiv-gnn.zip
%cd molhiv-gnn
!ls

## 5. Record the environment

Writes the exact versions into `results/` so any number produced below can be
traced back to the environment that produced it.

In [ ]:
import json, os, platform, subprocess, sys
os.makedirs('results/logs', exist_ok=True)

import torch, torch_geometric, ogb, rdkit, sklearn, numpy, scipy, pandas, ot
env = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'torch_geometric': torch_geometric.__version__,
    'ogb': ogb.__version__,
    'rdkit': rdkit.__version__,
    'scikit-learn': sklearn.__version__,
    'numpy': numpy.__version__,
    'scipy': scipy.__version__,
    'pandas': pandas.__version__,
    'POT': ot.__version__,
}
with open('results/logs/environment.json', 'w') as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))

## 6. Load the data and sanity-check the split

The zip already carries `dataset/ogbg_molhiv/` with `raw/`, `split/` and
`mapping/` (~2 MB), so this rebuilds the processed graphs locally in a few
seconds rather than downloading from `snap.stanford.edu`. Nothing here needs the
network, which also means the run does not depend on a single academic host
staying up.

This is also the cell that catches the most common breakage: `ogb` 1.3.6 calls
`torch.load` without `weights_only`, and torch >= 2.6 defaults that to `True`,
which makes the dataset unloadable. `src/data.py` allowlists the three PyG
classes involved, so this should just work.

The printed numbers must be **41127 / 32901 / 4113 / 4113** and **3.509%**.

In [ ]:
from src.data import load_dataset
dataset, split_idx = load_dataset()
print(dataset)
print('splits        :', {k: len(v) for k, v in split_idx.items()})
print('molecules     :', len(dataset))
print('positives     :', int(dataset.y.sum()))
print('positive rate : %.5f' % float(dataset.y.float().mean()))

assert len(dataset) == 41127, len(dataset)
assert [len(split_idx[k]) for k in ('train', 'valid', 'test')] == [32901, 4113, 4113]
print('\nsplit matches the official scaffold split.')

## 7. Arm 1 — Morgan fingerprints + Random Forest

The chemistry-first baseline: count substructures, feed them to a forest. CPU,
a few minutes. **If the GNN cannot beat this, that is the finding.**

7 of molhiv's SMILES do not parse in RDKit (3 train, 2 valid, 2 test, all
inactive). They are dropped from *training* only; the 2 test molecules are kept
with an all-zero fingerprint so this arm is scored on the same 4,113 molecules
as the other two.

In [ ]:
%%time
!python -u -m src.baseline_fp --seeds 0 1 2 --n-jobs {NJOBS} 2>&1 | tee results/logs/arm1_fp_rf.log

## 8. Arm 2 — WEGL (graph structure, no learning)

Diffuse atom features over bonds, treat each molecule as a point cloud, embed it
by optimal transport against one shared reference cloud, then fit a classical
classifier on top. No weights are learned by the embedding.

The PCA and the reference cloud are fitted on **training molecules only** and
applied unchanged to valid/test. The embedding is rebuilt per seed, so the
reported std covers the random reference-cloud draw instead of hiding it.

About 2 min of embedding plus ~90 s of forest per seed.

`--no-cache` skips the embedding cache, which is ~150 MB per seed and only ~2 min
to rebuild. Drop the flag if you want to try another classifier on a
byte-identical embedding.

In [ ]:
%%time
!python -u -m src.run_wegl --seeds 0 1 2 --classifier rf --no-cache --n-jobs {NJOBS} 2>&1 | tee results/logs/arm2_wegl.log

### Why a Random Forest here, and not boosted trees (optional ablation)

WEGL's embedding is `reference_size x pca_dim` = 30 x 32 = **960 dense float
features**. Histogram boosting scans every feature at every split, which on this
embedding measures at **12.1 s per boosting iteration** (7.8 s/iter at 240 dims,
16.6 s/iter at 480 — cost is roughly linear in width), so a full boosted run is
*hours* per seed. A Random Forest samples about 31 of the 960 features per split
and fits all 500 trees in **89 s**.

What arm 2 is testing is the **embedding** — structure without learning — not the
downstream classifier, so the cheap classifier is the right default. The cell
below runs the boosted version anyway, to confirm the conclusion does not hinge
on that choice. It writes to a separate folder so it cannot overwrite the
headline predictions.

**This is the expensive cell in the notebook.** `--max-iter 100` with patience
bounds it; without a cap it runs for hours. Skip it on a first pass.

In [ ]:
%%time
!python -u -m src.run_wegl --seeds 0 --classifier hgb --no-cache --n-jobs {NJOBS} --max-iter 100 --sweep-patience 2 --out results_wegl_hgb 2>&1 | tee results/logs/arm2_wegl_hgb.log

## 9. Arm 3 — PNA (trained GNN)

~25 min per seed on a T4, so ~75 min for three. Predictions and a checkpoint are
written after each seed.

The test score is read at the **best validation epoch**, never the best test
epoch. The degree histogram that PNA's scalers need is computed on **training
molecules only** — using all data would leak test structure into the
normalisation.

In [ ]:
%%time
!python -u -m src.train --seeds 0 1 2 --epochs 50 2>&1 | tee results/logs/arm3_pna.log

## 10. The comparison table

Prints mean ± std over seeds, a bootstrap 95% CI on the test molecules, and the
per-seed scores. It also prints each arm's test-set size and positive count and
**refuses to call the table comparable if they differ** — that guard is the whole
point of the exercise.

In [ ]:
!python -u -m src.report 2>&1 | tee results/logs/report.log

### How to read it

Two different uncertainties, and they answer different questions:

- **seed-to-seed std** — how stable the training is.
- **bootstrap 95% CI** — how much the score depends on *which molecules* landed
  in the test set. With 4,113 test molecules and only ~144 actives, expect this
  to be much wider than the std.

Do **not** conclude "indistinguishable" just because two arms' CIs overlap.
Those are *marginal* intervals, and both arms are scored on the same molecules,
so an unlucky draw of hard molecules moves both at once. Overlapping marginal
CIs are consistent with a real difference. The next cell does the test that
actually answers it.

### Is the gap bigger than the noise? (paired bootstrap)

The question "is A better than B" is about the *difference*, which has its own
and usually narrower distribution. `src.compare` resamples molecules once per
draw, scores both arms on that same resample, and reports a CI for the
difference plus the fraction of resamples in which A wins.

For reference, the two CPU arms measured locally: fingerprints lead WEGL by
**+0.0148**, yet every per-seed interval still includes zero, and the ordering
survives only 74–81% of resampled test sets. With 4,113 test molecules and 130
actives, a gap that size is **not resolvable** — which is a legitimate result and
a more interesting one than a ranking the data cannot support.

Watch whether arm 3 clears that bar. Beating fingerprints by a tenth of a point
would be unambiguous; beating them by 0.015 would not be.

In [ ]:
!python -u -m src.compare 2>&1 | tee results/logs/compare.log

## 11. Error analysis

The most confident mistakes, drawn. This is the part a chemist will actually
look at, and the question that follows the metrics.

In [ ]:
!python -u -m src.error_analysis --preds results/test_preds_seed0.npz 2>&1 | tee results/logs/error_analysis.log
from IPython.display import Image, display
for name in ('false_positives.png', 'false_negatives.png'):
    print(name)
    display(Image('results/' + name))

## 12. Calibration

ROC-AUC only cares about ranking. This asks a different question: when the model
says 0.8, does that behave like 0.8? Brier score plus a reliability curve.

In [ ]:
import glob, json
for path in sorted(glob.glob('results/calibration_seed*.json')):
    d = json.load(open(path))
    print(path, '-> Brier %.5f' % d['brier'])
    for frac, mean in zip(d['fraction_of_positives'], d['mean_predicted_value']):
        print('   predicted %.4f   actual %.4f' % (mean, frac))

## 13. Imbalance experiment (optional)

molhiv is 3.5% positive. The headline runs above use a plain unweighted loss, so
the main numbers are not tuned on this. Does weighting the positive class help?
Written to a separate folder so it cannot contaminate the table.

Oversampling is deliberately *not* the comparison here: on a scaffold split,
duplicating actives can put the same scaffold on both sides.

In [ ]:
%%time
!python -u -m src.train --seeds 0 --epochs 50 --pos-weight 10 --out results_posweight 2>&1 | tee results/logs/arm3_posweight.log

## 14. Download the results

Excludes the `.pkl` embedding caches and the `.pt` checkpoints, which are large
and regenerable. Keep the `.npz` predictions — `src.report` rebuilds the whole
table from those alone, with no retraining.

In [ ]:
import os
extra = ' '.join(d for d in ('results_posweight', 'results_wegl_hgb')
                 if os.path.isdir(d))
print('including:', 'results', extra or '(no optional folders)')
!zip -qr results.zip results {extra} -x '*.pkl' -x '*.pt'
!du -h results.zip
!unzip -l results.zip | tail -5
from google.colab import files
files.download('results.zip')

## 15. Serving check (optional)

SMILES in, probability out, plus the spread across the three seed checkpoints as
a cheap uncertainty estimate. Needs arm 3 to have produced `results/pna_seed*.pt`.

In [ ]:
import threading, time, requests, uvicorn
from app.main import app
threading.Thread(
    target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'),
    daemon=True).start()
time.sleep(10)
print(requests.get('http://127.0.0.1:8000/health').json())
print(requests.post('http://127.0.0.1:8000/predict',
                    json={'smiles': ['CC(=O)Oc1ccccc1C(=O)O', 'CCO']}).json())